# exp_018b - Targeted Merge Benchmark

Goal:
- benchmark a taxon-targeted merge where the `exp_018a` specialist only affects `Amphibia + Insecta` columns
- use pooled local OOF from `exp_011` as the generic baseline proxy
- estimate whether a later Kaggle overlay on top of `exp_015d` is worth attempting

Important limitation:
- this is a local proxy benchmark on aligned pooled OOF rows
- it is not a direct validation of `exp_015d`, because local `exp_015d` OOF artifacts are not available in the repo


In [ ]:
from __future__ import annotations

import json
import typing as tp
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

try:
    from IPython.display import display
except Exception:
    def display(obj: object) -> None:
        print(obj)


def resolve_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'PROJECT_STATE.md').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not resolve repository root')


@dataclass
class Config:
    experiment_id: str = 'exp_018b'
    experiment_name: str = 'targeted_merge_benchmark'
    target_taxa: tuple[str, ...] = ('Amphibia', 'Insecta')
    weight_grid: tuple[float, ...] = tuple(round(x, 2) for x in np.linspace(0.0, 1.0, 21))
    overall_weight_grid: tuple[float, ...] = (0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.75, 1.0)
    generic_experiment: str = 'exp_011_hgnetv2_soundscape_supervised'
    specialist_experiment: str = 'exp_018a_texture_specialist_oof'


CFG = Config()
ROOT = resolve_repo_root()
DATA = ROOT / 'data' / 'birdclef-2026'
OUTPUT_DIR = ROOT / 'experiments' / 'outputs' / f'{CFG.experiment_id}_{CFG.experiment_name}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_json(payload: dict[str, tp.Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=False))


print({'root': str(ROOT), 'output_dir': str(OUTPUT_DIR), 'target_taxa': CFG.target_taxa})


In [ ]:
def load_pooled_outputs(experiment_dir: Path) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    meta_parts: list[pd.DataFrame] = []
    probs_parts: list[np.ndarray] = []
    labels_parts: list[np.ndarray] = []
    fold_dirs = sorted(path for path in experiment_dir.iterdir() if path.is_dir() and path.name.startswith('fold_'))
    if not fold_dirs:
        raise FileNotFoundError(f'No fold directories found in {experiment_dir}')
    for fold_dir in fold_dirs:
        meta = pd.read_csv(fold_dir / 'best_valid_meta.csv')
        arr = np.load(fold_dir / 'best_valid_outputs.npz', allow_pickle=True)
        meta['fold_src'] = fold_dir.name
        meta_parts.append(meta)
        probs_parts.append(arr['probs'])
        labels_parts.append(arr['labels'])
    return pd.concat(meta_parts, ignore_index=True), np.concatenate(probs_parts, axis=0), np.concatenate(labels_parts, axis=0)


taxonomy = pd.read_csv(DATA / 'taxonomy.csv')
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
generic_classes = taxonomy['primary_label'].tolist()
generic_label2idx = {label: idx for idx, label in enumerate(generic_classes)}
taxon_lookup = taxonomy.set_index('primary_label')['class_name'].to_dict()

specialist_dir = ROOT / 'experiments' / 'outputs' / CFG.specialist_experiment
generic_dir = ROOT / 'experiments' / 'outputs' / CFG.generic_experiment
meta_spec, probs_spec, labels_spec = load_pooled_outputs(specialist_dir)
meta_gen, probs_gen, labels_gen = load_pooled_outputs(generic_dir)

target_config = json.loads((specialist_dir / 'fold_00' / 'target_config.json').read_text())
target_classes = [str(label) for label in target_config['target_classes']]
target_to_generic = [generic_label2idx[label] for label in target_classes]
target_taxon_map = {label: taxon_lookup.get(label, 'Unknown') for label in target_classes}

print({
    'specialist_rows': int(len(meta_spec)),
    'generic_rows': int(len(meta_gen)),
    'target_classes': int(len(target_classes)),
    'soundscape_specialist_rows': int(meta_spec['source'].astype(str).eq('soundscape_clip').sum()),
})


In [ ]:
ALIGN_KEYS = ['filename', 'source', 'clip_start_frame', 'clip_end_frame', 'primary_label']


def prep_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df[ALIGN_KEYS].copy()
    out['primary_label'] = out['primary_label'].astype(str)
    out['clip_start_frame'] = out['clip_start_frame'].fillna(-1).astype(int)
    out['clip_end_frame'] = out['clip_end_frame'].fillna(-1).astype(int)
    return out


keys_spec = prep_keys(meta_spec)
keys_gen = prep_keys(meta_gen)
keys_spec['_spec_idx'] = np.arange(len(keys_spec))
keys_gen['_gen_idx'] = np.arange(len(keys_gen))

aligned = keys_spec.merge(keys_gen, on=ALIGN_KEYS, how='inner')
assert len(aligned) == len(meta_spec), (len(aligned), len(meta_spec))

spec_idx = aligned['_spec_idx'].to_numpy()
gen_idx = aligned['_gen_idx'].to_numpy()

meta_aligned = meta_spec.iloc[spec_idx].reset_index(drop=True)
y_spec = labels_spec[spec_idx]
p_spec = probs_spec[spec_idx]
y_gen_target = labels_gen[gen_idx][:, target_to_generic]
p_gen_target = probs_gen[gen_idx][:, target_to_generic]
y_gen_all = labels_gen[gen_idx]
p_gen_all = probs_gen[gen_idx].copy()

sound_mask = meta_aligned['source'].astype(str).eq('soundscape_clip').to_numpy()

setup_snapshot = {
    'experiment_id': CFG.experiment_id,
    'experiment_name': CFG.experiment_name,
    'target_taxa': list(CFG.target_taxa),
    'target_classes': len(target_classes),
    'aligned_rows': int(len(meta_aligned)),
    'aligned_soundscape_rows': int(sound_mask.sum()),
    'generic_experiment': CFG.generic_experiment,
    'specialist_experiment': CFG.specialist_experiment,
}
save_json(setup_snapshot, OUTPUT_DIR / 'setup_snapshot.json')
print(json.dumps(setup_snapshot, indent=2))


In [ ]:
def macro_auc_skip_empty(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, int]:
    auc_values: list[float] = []
    for idx in range(y_true.shape[1]):
        yt = y_true[:, idx]
        if len(np.unique(yt)) < 2:
            continue
        auc_values.append(float(roc_auc_score(yt, y_score[:, idx])))
    return float(np.mean(auc_values)), int(len(auc_values))


generic_target_macro_auc, generic_target_scored = macro_auc_skip_empty(y_gen_target, p_gen_target)
generic_target_soundscape_auc, generic_target_soundscape_scored = macro_auc_skip_empty(y_gen_target[sound_mask], p_gen_target[sound_mask])
specialist_target_macro_auc, specialist_target_scored = macro_auc_skip_empty(y_spec, p_spec)
specialist_target_soundscape_auc, specialist_target_soundscape_scored = macro_auc_skip_empty(y_spec[sound_mask], p_spec[sound_mask])

baseline_df = pd.DataFrame([
    {
        'variant': 'generic_exp011_target_only',
        'target_macro_auc': generic_target_macro_auc,
        'target_soundscape_macro_auc': generic_target_soundscape_auc,
        'scored_classes': generic_target_scored,
        'soundscape_scored_classes': generic_target_soundscape_scored,
    },
    {
        'variant': 'specialist_exp018a_target_only',
        'target_macro_auc': specialist_target_macro_auc,
        'target_soundscape_macro_auc': specialist_target_soundscape_auc,
        'scored_classes': specialist_target_scored,
        'soundscape_scored_classes': specialist_target_soundscape_scored,
    },
])
display(baseline_df)


In [ ]:
weight_rows: list[dict[str, tp.Any]] = []
for w_spec in CFG.weight_grid:
    blend = (1.0 - w_spec) * p_gen_target + w_spec * p_spec
    target_macro_auc, scored_classes = macro_auc_skip_empty(y_gen_target, blend)
    target_soundscape_macro_auc, soundscape_scored_classes = macro_auc_skip_empty(y_gen_target[sound_mask], blend[sound_mask])
    weight_rows.append({
        'w_spec': float(w_spec),
        'w_generic': float(1.0 - w_spec),
        'target_macro_auc': target_macro_auc,
        'target_soundscape_macro_auc': target_soundscape_macro_auc,
        'scored_classes': scored_classes,
        'soundscape_scored_classes': soundscape_scored_classes,
    })

weight_sweep = pd.DataFrame(weight_rows).sort_values(['target_soundscape_macro_auc', 'target_macro_auc'], ascending=[False, False]).reset_index(drop=True)
best_weight = float(weight_sweep.iloc[0]['w_spec'])
best_blend = (1.0 - best_weight) * p_gen_target + best_weight * p_spec

print('Top target-only merge candidates:')
display(weight_sweep.head(12))


In [ ]:
def macro_auc_all(y_true: np.ndarray, y_score: np.ndarray) -> tuple[float, int]:
    auc_values: list[float] = []
    for idx in range(y_true.shape[1]):
        yt = y_true[:, idx]
        if len(np.unique(yt)) < 2:
            continue
        auc_values.append(float(roc_auc_score(yt, y_score[:, idx])))
    return float(np.mean(auc_values)), int(len(auc_values))


overall_baseline_auc, overall_scored = macro_auc_all(y_gen_all, p_gen_all)
overall_rows: list[dict[str, tp.Any]] = []
for w_spec in CFG.overall_weight_grid:
    merged_probs = p_gen_all.copy()
    merged_probs[:, target_to_generic] = (1.0 - w_spec) * merged_probs[:, target_to_generic] + w_spec * p_spec
    overall_macro_auc_aligned_subset, overall_scored_classes = macro_auc_all(y_gen_all, merged_probs)
    target_macro_auc, _ = macro_auc_skip_empty(y_gen_target, merged_probs[:, target_to_generic])
    target_soundscape_macro_auc, _ = macro_auc_skip_empty(y_gen_target[sound_mask], merged_probs[sound_mask][:, target_to_generic])
    overall_rows.append({
        'w_spec': float(w_spec),
        'w_generic': float(1.0 - w_spec),
        'overall_macro_auc_aligned_subset': overall_macro_auc_aligned_subset,
        'target_macro_auc': target_macro_auc,
        'target_soundscape_macro_auc': target_soundscape_macro_auc,
        'overall_scored_classes': overall_scored_classes,
    })

overall_merge_sweep = pd.DataFrame(overall_rows).sort_values('target_soundscape_macro_auc', ascending=False).reset_index(drop=True)
display(overall_merge_sweep)

taxon_rows: list[dict[str, tp.Any]] = []
target_taxon_array = np.array([target_taxon_map[label] for label in target_classes], dtype=object)
for taxon in sorted(set(target_taxon_array.tolist())):
    taxon_mask = target_taxon_array == taxon
    generic_auc, generic_sc = macro_auc_skip_empty(y_gen_target[:, taxon_mask], p_gen_target[:, taxon_mask])
    specialist_auc, specialist_sc = macro_auc_skip_empty(y_spec[:, taxon_mask], p_spec[:, taxon_mask])
    blend_auc, blend_sc = macro_auc_skip_empty(y_gen_target[:, taxon_mask], best_blend[:, taxon_mask])
    generic_sound_auc, generic_sound_sc = macro_auc_skip_empty(y_gen_target[sound_mask][:, taxon_mask], p_gen_target[sound_mask][:, taxon_mask])
    specialist_sound_auc, specialist_sound_sc = macro_auc_skip_empty(y_spec[sound_mask][:, taxon_mask], p_spec[sound_mask][:, taxon_mask])
    blend_sound_auc, blend_sound_sc = macro_auc_skip_empty(y_gen_target[sound_mask][:, taxon_mask], best_blend[sound_mask][:, taxon_mask])
    taxon_rows.append({
        'taxon': taxon,
        'generic_target_macro_auc': generic_auc,
        'specialist_target_macro_auc': specialist_auc,
        'best_merge_target_macro_auc': blend_auc,
        'generic_target_soundscape_macro_auc': generic_sound_auc,
        'specialist_target_soundscape_macro_auc': specialist_sound_auc,
        'best_merge_target_soundscape_macro_auc': blend_sound_auc,
        'scored_classes': generic_sc,
        'soundscape_scored_classes': generic_sound_sc,
    })
taxon_summary = pd.DataFrame(taxon_rows).sort_values('best_merge_target_soundscape_macro_auc', ascending=False).reset_index(drop=True)
display(taxon_summary)


In [ ]:
weight_sweep.to_csv(OUTPUT_DIR / 'weight_sweep.csv', index=False)
overall_merge_sweep.to_csv(OUTPUT_DIR / 'overall_merge_sweep.csv', index=False)
taxon_summary.to_csv(OUTPUT_DIR / 'taxon_summary.csv', index=False)
baseline_df.to_csv(OUTPUT_DIR / 'baseline_summary.csv', index=False)

report_snapshot = {
    'experiment_id': CFG.experiment_id,
    'experiment_name': CFG.experiment_name,
    'target_taxa': list(CFG.target_taxa),
    'aligned_rows': int(len(meta_aligned)),
    'aligned_soundscape_rows': int(sound_mask.sum()),
    'generic_target_macro_auc': float(generic_target_macro_auc),
    'generic_target_soundscape_macro_auc': float(generic_target_soundscape_auc),
    'specialist_target_macro_auc': float(specialist_target_macro_auc),
    'specialist_target_soundscape_macro_auc': float(specialist_target_soundscape_auc),
    'best_weight_target_only': best_weight,
    'best_target_macro_auc': float(weight_sweep.iloc[0]['target_macro_auc']),
    'best_target_soundscape_macro_auc': float(weight_sweep.iloc[0]['target_soundscape_macro_auc']),
    'overall_aligned_subset_baseline': float(overall_baseline_auc),
    'best_overall_merge_weight_in_grid': float(overall_merge_sweep.iloc[0]['w_spec']),
    'best_overall_aligned_subset_macro_auc': float(overall_merge_sweep.iloc[0]['overall_macro_auc_aligned_subset']),
    'proxy_note': 'Generic baseline is pooled exp_011 OOF; use this as a direction finder before any Kaggle overlay on exp_015d.',
}
save_json(report_snapshot, OUTPUT_DIR / 'report_snapshot.json')
print(json.dumps(report_snapshot, indent=2))
